# DX 704 Week 6 Project

This project will develop a treatment plan for a fictious illness "Twizzleflu".
Twizzleflu is a mild illness caused by a virus.
The main symptoms are a mild fever, fidgeting, and kicking the blankets off the bed or couch.
Mild dehydration has also been reported in more severe cases.
These symptoms typically last 1-2 weeks without treatment.
Word on the internet says that Twizzleflu can be cured faster by drinking copious orange juice, but this has not been supported by evidence so far.
You will be provided with a theoretical model of Twizzleflu modeled as a Markov decision process.
Based on the model, you will compute optimal treatment plans to optimize different criteria, and compare patient discomfort with the different plans.

The full project description, a template notebook, and raw data are available on GitHub: [Project 6 Materials](https://github.com/bu-cds-dx704/dx704-project-06).

We will model Twizzleflu as a Markov decision process.
The model transition probabilities are provided in the file "twizzleflu-transitions.tsv" and the expected rewards are in "twizzleflu-rewards.tsv".
The goal for Twizzleflu is to minimize the expected discomfort of the patient which is expressed as negative rewards in the file.

## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Evaluate a Do Nothing Plan

One of the treatment actions is to do nothing.
Calculate the expected discomfort (not rewards) of a policy that always does nothing.

Hint: for this value calculation and later ones, use value iteration.
The analytical solution has difficulties in practice when there is no discount factor.

In [1]:
# YOUR CHANGES HERE

import pandas as pd
import numpy as np

# Load data
transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
rewards = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")

# Policy: always do-nothing
ACTION = "do-nothing"
terminal = "recovered"

# Filter to do-nothing policy
t = transitions[transitions["action"] == ACTION].copy()
r = rewards[rewards["action"] == ACTION].copy()

# States and indexing
states = sorted(t["state"].unique())
idx = {s: i for i, s in enumerate(states)}
S = len(states)

# Transition matrix P under the policy
P = np.zeros((S, S), dtype=float)
for _, row in t.iterrows():
    s = row["state"]
    s2 = row["next_state"]
    p = float(row["probability"])
    P[idx[s], idx[s2]] += p

# Reward vector R under the policy (expected immediate reward per state)
r_map = dict(zip(r["state"], r["reward"]))
R = np.array([float(r_map.get(s, 0.0)) for s in states], dtype=float)

# Solve undiscounted value function with terminal fixed at 0:
# V = R + P V  => (I - P) V = R, but enforce V[terminal] = 0 by solving only nonterminal part
if terminal not in idx:
    raise ValueError(f"Terminal state '{terminal}' not found in states: {states}")

nonterminal_states = [s for s in states if s != terminal]
nt_idx = [idx[s] for s in nonterminal_states]

P_tt = P[np.ix_(nt_idx, nt_idx)]
R_t  = R[nt_idx]

V_t = np.linalg.solve(np.eye(len(nt_idx)) - P_tt, R_t)

V = np.zeros(S, dtype=float)
V[nt_idx] = V_t
V[idx[terminal]] = 0.0

# Expected discomfort is negative expected reward (rewards are negative for discomfort)
df_out = pd.DataFrame({
    "state": states,
    "expected_discomfort": -V
}).sort_values("state")

# Clean up negative zero / tiny negatives
df_out["expected_discomfort"] = df_out["expected_discomfort"].clip(lower=0)

# Display and save
df_out.to_csv("do-nothing-discomfort.tsv", sep="\t", index=False)
print("Saved: do-nothing-discomfort.tsv")

df_out

Saved: do-nothing-discomfort.tsv


,state,expected_discomfort
0,exposed-1,3.413333
1,exposed-2,4.266667
2,exposed-3,5.333333
3,recovered,-0.000000
4,symptoms-1,6.666667
5,symptoms-2,5.000000
6,symptoms-3,1.666667


Save the expected discomfort by state to a file "do-nothing-discomfort.tsv" with columns state and expected_discomfort.

In [2]:
# YOUR CHANGES HERE

df_out.to_csv("do-nothing-discomfort.tsv", sep="\t", index=False)
print("Saved: do-nothing-discomfort.tsv") 

Saved: do-nothing-discomfort.tsv


Submit "do-nothing-discomfort.tsv" in Gradescope.

## Part 2: Compute an Optimal Treatment Plan

Compute an optimal treatment plan for Twizzleflu.
It should minimize the expected discomfort (maximize the rewards).

Save the optimal actions for each state to a file "minimum-discomfort-actions.tsv" with columns state and action.

In [3]:
# YOUR CHANGES HERE

import pandas as pd
import numpy as np

transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
rewards = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")

states = list(pd.unique(transitions["state"]))      # keep file order
actions = list(pd.unique(transitions["action"]))    # keep file order

s_idx = {s:i for i,s in enumerate(states)}
a_idx = {a:i for i,a in enumerate(actions)}

S, A = len(states), len(actions)

# P[a, s, s2]
P = np.zeros((A, S, S), dtype=float)
for _, row in transitions.iterrows():
    a = a_idx[row["action"]]
    s = s_idx[row["state"]]
    s2 = s_idx[row["next_state"]]
    P[a, s, s2] += float(row["probability"])

# R[a, s]  (reward is for CURRENT state)
R = np.zeros((A, S), dtype=float)
for _, row in rewards.iterrows():
    a = a_idx[row["action"]]
    s = s_idx[row["state"]]
    R[a, s] = float(row["reward"])

# Value Iteration
gamma = 0.99
theta = 1e-12
max_iter = 100000

V = np.zeros(S, dtype=float)
for _ in range(max_iter):
    V_old = V.copy()
    Q = R + gamma * np.einsum("ast,t->as", P, V_old)   # <-- FIXED einsum
    V = np.max(Q, axis=0)
    if np.max(np.abs(V - V_old)) < theta:
        break

pi = np.argmax(R + gamma * np.einsum("ast,t->as", P, V), axis=0)

df_actions = pd.DataFrame({
    "state": states,
    "action": [actions[a] for a in pi]
})

df_actions

,state,action
0,exposed-1,sleep-8
1,exposed-2,sleep-8
2,exposed-3,sleep-8
3,symptoms-1,drink-oj
4,symptoms-2,drink-oj
5,symptoms-3,drink-oj
6,recovered,do-nothing


In [4]:
# YOUR CHANGES HERE

df_actions = df_actions[["state", "action"]]
df_actions.to_csv("minimum-discomfort-actions.tsv", sep="\t", index=False)
print("Saved: minimum-discomfort-actions.tsv")

Saved: minimum-discomfort-actions.tsv


Submit "minimum-discomfort-actions.tsv" in Gradescope.

## Part 3: Expected Discomfort

Using your previous optimal policy, compute the expected discomfort for each state.

In [5]:
# YOUR CHANGES HERE

import pandas as pd
import numpy as np

# Load MDP
transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
rewards = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")

states = sorted(transitions["state"].unique())
actions = sorted(transitions["action"].unique())

s_idx = {s: i for i, s in enumerate(states)}
a_idx = {a: i for i, a in enumerate(actions)}

S, A = len(states), len(actions)

# Build P[a, s, s']
P = np.zeros((A, S, S), dtype=float)
for _, row in transitions.iterrows():
    a = a_idx[row["action"]]
    s = s_idx[row["state"]]
    s2 = s_idx[row["next_state"]]
    P[a, s, s2] += float(row["probability"])

# Build R[a, s]
R = np.zeros((A, S), dtype=float)
for _, row in rewards.iterrows():
    a = a_idx[row["action"]]
    s = s_idx[row["state"]]
    R[a, s] = float(row["reward"])

# Load the policy computed in Part 2
df_actions = pd.read_csv("minimum-discomfort-actions.tsv", sep="\t")
pi = np.array([a_idx[df_actions.loc[df_actions["state"] == s, "action"].iloc[0]] for s in states], dtype=int)

# Policy-specific transition matrix P_pi and reward vector R_pi
P_pi = np.zeros((S, S), dtype=float)
R_pi = np.zeros(S, dtype=float)

for s in states:
    si = s_idx[s]
    a = pi[si]
    P_pi[si, :] = P[a, si, :]
    R_pi[si] = R[a, si]

# Solve undiscounted expected return V_pi with terminal fixed at 0
terminal = "recovered"
nt_states = [s for s in states if s != terminal]
nt_idx = [s_idx[s] for s in nt_states]

P_tt = P_pi[np.ix_(nt_idx, nt_idx)]
R_t  = R_pi[nt_idx]

V_t = np.linalg.solve(np.eye(len(nt_idx)) - P_tt, R_t)

V = np.zeros(S, dtype=float)
V[nt_idx] = V_t
V[s_idx[terminal]] = 0.0

# Convert to expected discomfort (negative of expected reward)
df_values = pd.DataFrame({
    "state": states,
    "expected_discomfort": (-V)
}).sort_values("state")

# Clean formatting (no negative zero)
df_values["expected_discomfort"] = df_values["expected_discomfort"].round(6)
df_values.loc[df_values["expected_discomfort"].abs() < 1e-12, "expected_discomfort"] = 0.0

df_values

,state,expected_discomfort
0,exposed-1,0.75
1,exposed-2,1.50
2,exposed-3,3.00
3,recovered,0.00
4,symptoms-1,6.00
5,symptoms-2,4.50
6,symptoms-3,1.50


Save your results in a file "minimum-discomfort-values.tsv" with columns state and expected_discomfort.

In [6]:
# YOUR CHANGES HERE

df_values = df_values[["state", "expected_discomfort"]]

df_values.to_csv("minimum-discomfort-values.tsv", sep="\t", index=False)
print("Saved: minimum-discomfort-values.tsv")

Saved: minimum-discomfort-values.tsv


Submit "minimum-discomfort-values.tsv" in Gradescope.

## Part 4: Minimizing Twizzleflu Duration

Modifiy the Markov decision process to minimize the days until the Twizzle flu is over.
To do so, change the reward function to always be -1 if the current state corresponds to being sick (must have symptoms, exposed does not count) and 0 otherwise.
To be clear, the action does not matter for this reward function.


Save your new reward function in a file "duration-rewards.tsv" in the same format as "twizzleflu-rewards.tsv".

In [7]:
# YOUR CHANGES HERE

import pandas as pd

transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")

states = list(pd.unique(transitions["state"]))      # keep file order
actions = list(pd.unique(transitions["action"]))    # keep file order

rows = []
for s in states:
    reward_val = -1.0 if str(s).startswith("symptoms") else 0.0
    for a in actions:
        rows.append({"action": a, "state": s, "reward": reward_val})

duration_rewards = pd.DataFrame(rows)

duration_rewards.to_csv("duration-rewards.tsv", sep="\t", index=False)
print("Saved: duration-rewards.tsv")

duration_rewards.head()

Saved: duration-rewards.tsv


,action,state,reward
0,do-nothing,exposed-1,0.0
1,drink-oj,exposed-1,0.0
2,sleep-8,exposed-1,0.0
3,do-nothing,exposed-2,0.0
4,drink-oj,exposed-2,0.0


In [8]:
duration_rewards.to_csv("duration-rewards.tsv", sep="\t", index=False)
print("Saved: duration-rewards.tsv")

Saved: duration-rewards.tsv


Submit "duration-rewards.tsv" in Gradescope.

## Part 5: Optimize for Shorter Twizzleflu

Compute an optimal policy to minimize the duration of Twizzleflu.

In [9]:
# YOUR CHANGES HERE

import pandas as pd
import numpy as np

# Load transitions + duration rewards (from Part 4)
transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
duration_rewards = pd.read_csv("duration-rewards.tsv", sep="\t")

states = list(pd.unique(transitions["state"]))
actions = list(pd.unique(transitions["action"]))

s_idx = {s:i for i,s in enumerate(states)}
a_idx = {a:i for i,a in enumerate(actions)}
S, A = len(states), len(actions)

# Build P[a, s, s']
P = np.zeros((A, S, S), dtype=float)
for _, row in transitions.iterrows():
    P[a_idx[row["action"]], s_idx[row["state"]], s_idx[row["next_state"]]] += float(row["probability"])

# Build R[a, s] from duration_rewards
R = np.zeros((A, S), dtype=float)
for _, row in duration_rewards.iterrows():
    R[a_idx[row["action"]], s_idx[row["state"]]] = float(row["reward"])

# Value Iteration (maximize reward == minimize sick days)
theta = 1e-12
V = np.zeros(S, dtype=float)

while True:
    V_old = V.copy()
    Q = R + np.einsum("ast,t->as", P, V_old)   # gamma = 1.0
    V = np.max(Q, axis=0)
    if np.max(np.abs(V - V_old)) < theta:
        break

pi = np.argmax(R + np.einsum("ast,t->as", P, V), axis=0)

df_time_actions = pd.DataFrame({
    "state": states,
    "action": [actions[a] for a in pi]
})

df_time_actions.to_csv("minimum-duration-actions.tsv", sep="\t", index=False)
print("Saved: minimum-duration-actions.tsv")

df_time_actions

Saved: minimum-duration-actions.tsv


,state,action
0,exposed-1,sleep-8
1,exposed-2,sleep-8
2,exposed-3,sleep-8
3,symptoms-1,do-nothing
4,symptoms-2,do-nothing
5,symptoms-3,do-nothing
6,recovered,do-nothing


Save the optimal actions for each state to a file "minimum-duration-actions.tsv" with columns state and action.

In [10]:
# YOUR CHANGES HERE

df_time_actions = df_time_actions[["state", "action"]]

df_time_actions.to_csv("minimum-duration-actions.tsv", sep="\t", index=False)
print("Saved: minimum-duration-actions.tsv") 

Saved: minimum-duration-actions.tsv


Submit "minimum-duration-actions.tsv" in Gradescope.

## Part 6: Shorter Twizzleflu?

Compute the expected number of days sick for each state to a file.

In [11]:
# YOUR CHANGES HERE

import pandas as pd
import numpy as np

# Load transitions + the minimum-duration policy
transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
df_pol = pd.read_csv("minimum-duration-actions.tsv", sep="\t")

states = list(pd.unique(transitions["state"]))
actions = list(pd.unique(transitions["action"]))

s_idx = {s:i for i,s in enumerate(states)}
a_idx = {a:i for i,a in enumerate(actions)}
S, A = len(states), len(actions)

# Build P[a, s, s']
P = np.zeros((A, S, S), dtype=float)
for _, row in transitions.iterrows():
    P[a_idx[row["action"]], s_idx[row["state"]], s_idx[row["next_state"]]] += float(row["probability"])

# Policy vector pi[s] = action index
act_for_state = dict(zip(df_pol["state"], df_pol["action"]))
pi = np.array([a_idx[act_for_state[s]] for s in states], dtype=int)

# Policy-specific transition matrix P_pi
P_pi = np.zeros((S, S), dtype=float)
for s in states:
    si = s_idx[s]
    P_pi[si, :] = P[pi[si], si, :]

# Cost per step = 1 if in symptoms-* else 0
c = np.array([1.0 if str(s).startswith("symptoms") else 0.0 for s in states], dtype=float)

# Solve E = c + P_pi E with E[recovered] = 0
terminal = "recovered"
nt_states = [s for s in states if s != terminal]
nt_idx = [s_idx[s] for s in nt_states]

E_t = np.linalg.solve(np.eye(len(nt_idx)) - P_pi[np.ix_(nt_idx, nt_idx)], c[nt_idx])

E = np.zeros(S, dtype=float)
E[nt_idx] = E_t
E[s_idx[terminal]] = 0.0

df_days = pd.DataFrame({
    "state": states,
    "expected_sick_days": E
})

df_days.to_csv("minimum-duration-days.tsv", sep="\t", index=False)
print("Saved: minimum-duration-days.tsv")

df_days

Saved: minimum-duration-days.tsv


,state,expected_sick_days
0,exposed-1,1.250000
1,exposed-2,2.500000
2,exposed-3,5.000000
3,symptoms-1,10.000000
4,symptoms-2,6.666667
5,symptoms-3,3.333333
6,recovered,0.000000


Save the expected sick days for each state to a file "minimum-duration-days.tsv" with columns state and expected_sick_days.

In [12]:
# YOUR CHANGES HERE

df_days = df_days[["state", "expected_sick_days"]]

df_days.to_csv("minimum-duration-days.tsv", sep="\t", index=False)
print("Saved: minimum-duration-days.tsv") 

Saved: minimum-duration-days.tsv


Submit "minimum-duration-days.tsv" in Gradescope.

## Part 7: Speed vs Pampering

Compute the expected discomfort using the policy to minimize days sick, and compare the results to the expected discomfort when optimizing to minimize discomfort.

In [13]:
# YOUR CHANGES HERE

import pandas as pd
import numpy as np

df_do = pd.read_csv("do-nothing-discomfort.tsv", sep="\t").rename(
    columns={"expected_discomfort": "do_nothing_discomfort"}
)

df_min = pd.read_csv("minimum-discomfort-values.tsv", sep="\t").rename(
    columns={"expected_discomfort": "minimize_discomfort"}
)

# --- Compute speed_discomfort (discomfort under minimum-duration policy) ---

transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
rewards = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")

states = list(pd.unique(transitions["state"]))
actions = list(pd.unique(transitions["action"]))

s_idx = {s:i for i,s in enumerate(states)}
a_idx = {a:i for i,a in enumerate(actions)}
S, A = len(states), len(actions)

# Build transition tensor P[a, s, s']
P = np.zeros((A, S, S), dtype=float)
for _, row in transitions.iterrows():
    P[a_idx[row["action"]], s_idx[row["state"]], s_idx[row["next_state"]]] += float(row["probability"])

# Build reward matrix R[a, s]
R = np.zeros((A, S), dtype=float)
for _, row in rewards.iterrows():
    R[a_idx[row["action"]], s_idx[row["state"]]] = float(row["reward"])

# Load minimum-duration policy
df_speed_pol = pd.read_csv("minimum-duration-actions.tsv", sep="\t")
act_for_state = dict(zip(df_speed_pol["state"], df_speed_pol["action"]))
pi = np.array([a_idx[act_for_state[s]] for s in states], dtype=int)

# Build policy-specific transition matrix and rewards
P_pi = np.zeros((S, S), dtype=float)
R_pi = np.zeros(S, dtype=float)

for s in states:
    si = s_idx[s]
    a = pi[si]
    P_pi[si, :] = P[a, si, :]
    R_pi[si] = R[a, si]

# Solve V = R + P V with terminal fixed at 0
terminal = "recovered"
nt_states = [s for s in states if s != terminal]
nt_idx = [s_idx[s] for s in nt_states]

V_t = np.linalg.solve(np.eye(len(nt_idx)) - P_pi[np.ix_(nt_idx, nt_idx)], R_pi[nt_idx])

V = np.zeros(S, dtype=float)
V[nt_idx] = V_t
V[s_idx[terminal]] = 0.0

df_speed = pd.DataFrame({
    "state": states,
    "speed_discomfort": -V
})

# --- Build final comparison dataframe ---
df_compare = df_do.merge(df_min, on="state").merge(df_speed, on="state")

# Ensure exact column order and names required by autograder
df_compare = df_compare[[
    "state",
    "do_nothing_discomfort",
    "minimize_discomfort",
    "speed_discomfort"
]]

df_compare.to_csv("policy-comparison.tsv", sep="\t", index=False)
print("Saved: policy-comparison.tsv")

df_compare

Saved: policy-comparison.tsv


,state,do_nothing_discomfort,minimize_discomfort,speed_discomfort
0,exposed-1,3.413333,0.75,0.833333
1,exposed-2,4.266667,1.50,1.666667
2,exposed-3,5.333333,3.00,3.333333
3,recovered,-0.000000,0.00,-0.000000
4,symptoms-1,6.666667,6.00,6.666667
5,symptoms-2,5.000000,4.50,5.000000
6,symptoms-3,1.666667,1.50,1.666667


Save the results to a file "policy-comparison.tsv" with columns state, speed_discomfort, and minimize_discomfort.

In [14]:
# YOUR CHANGES HERE

df_compare = df_compare[[
    "state",
    "do_nothing_discomfort",
    "minimize_discomfort",
    "speed_discomfort"
]]

df_compare.to_csv("policy-comparison.tsv", sep="\t", index=False)
print("Saved: policy-comparison.tsv") 

Saved: policy-comparison.tsv


Submit "policy-comparison.tsv" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

In [15]:
ack_text = """Acknowledgements

I completed this assignment independently.

Libraries used: numpy and pandas for reading/writing TSV files and performing matrix and linear algebra computations for policy evaluation and optimization.

Generative AI: I used ChatGPT to clarify concepts and assist with debugging implementation details related to Markov decision processes, value iteration, and policy evaluation. I reviewed, adapted, and verified all code myself before submission.
"""

filename = "acknowledgements.txt"

with open(filename, "w") as f:
    f.write(ack_text)

print(f"Saved: {filename}")

Saved: acknowledgements.txt
